# 32 GPU deep CV feature extraction

L4 推奨。YOLO detection/tracking、pose skeleton、bat/plate/homography、確認用 overlay video をまとめます。

各 stage は `BASE_DIR/reports/preflight/compact_runs/` に JSONL log を追記します。既に期待 artifact が揃っている stage は skip します。再実行したい場合は `FORCE_RERUN_ALL=True` にしてください。

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


# v2 をデフォルトにします。v1 を再実行したい時だけここを書き換えてください。
os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください。\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は compact notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
NOTEBOOK_DIR = REPO_DIR / 'notebooks'
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ['BASEBALL_VISION_RUN_PROFILE']
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.compact_notebooks import CompactRunLogger
from sport_pipeline.io import read_table
from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, run_id, stage_settings

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
SEQUENCE_RUN_ID = run_id(RUN_PROFILE, 'sequence_run_id')
SEQUENCE_TCN_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
VIDEO_LIGHTWEIGHT_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id')
VIDEO_FROZEN_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id')
VIDEO_FINETUNE_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id')
PLAYER_SEASON_RUN_ID = run_id(RUN_PROFILE, 'player_season_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id')
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
METHOD_EVALUATION_REPORT_ID = run_id(RUN_PROFILE, 'method_evaluation_report_id')
VIDEO_ABLATION_REPORT_ID = run_id(RUN_PROFILE, 'video_ablation_report_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
CLIP_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'clip_embedding_feature_id')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')
VIDEO_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_embedding_feature_id')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')

print('RUN_PROFILE =', RUN_PROFILE_NAME)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('BASE_DIR =', BASE_DIR)
print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('src_dir =', src_dir)


In [ ]:
COMPACT_RUN_NAME = '32_gpu_deep_cv_feature_extraction'
LOGGER = CompactRunLogger(BASE_DIR, COMPACT_RUN_NAME, run_profile_name=RUN_PROFILE_NAME)
FORCE_RERUN_ALL = False

DEEP_CV_SETTINGS = stage_settings(RUN_PROFILE, 'deep_cv')
ENABLE_RTMPOSE = bool(DEEP_CV_SETTINGS.get('enable_rtmpose', False))
POSE_BACKEND = str(DEEP_CV_SETTINGS.get('pose_backend', 'rtmpose')).lower()
DEEP_CV_PROGRESS_PATH = BASE_DIR / 'reports' / 'preflight' / f'deep_cv_{FULL_RUN_ID}_progress.json'
POSE2D_PATH = BASE_DIR / 'pose2d' / FULL_RUN_ID / 'pose2d_v1.parquet'
STRUCTURED_FRAMES_PATH = BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'frames.parquet'
OVERLAY_MANIFEST_PATH = BASE_DIR / 'debug' / FULL_RUN_ID / 'cv_overlay_videos_v1.parquet'

def _safe_read_rows(path):
    try:
        return read_table(path) if path.exists() else []
    except Exception as exc:
        print('WARNING: completion check failed for', path, exc)
        return []

def _safe_read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    except Exception as exc:
        print('WARNING: progress check failed for', path, exc)
        return {}

def _frame_pose_coverage(row):
    if row.get('pose_coverage') is not None:
        return float(row.get('pose_coverage') or 0.0)
    names = row.get('feature_names') or []
    values = row.get('feature_values') or []
    if isinstance(names, list) and isinstance(values, list) and 'pose_coverage' in names:
        index = names.index('pose_coverage')
        if index < len(values):
            return float(values[index] or 0.0)
    return 0.0

deep_cv_progress_before = _safe_read_json(DEEP_CV_PROGRESS_PATH)
deep_cv_status_before = str(deep_cv_progress_before.get('status') or 'missing')
deep_cv_rows_before = deep_cv_progress_before.get('rows') or {}
deep_cv_resolved_clips_before = int(deep_cv_progress_before.get('resolved_clip_files') or 0)
deep_cv_completed_pose_before = int(deep_cv_progress_before.get('completed_pose_clips') or 0)
pose_rows_before = len(_safe_read_rows(POSE2D_PATH))
structured_frame_rows_before = _safe_read_rows(STRUCTURED_FRAMES_PATH)
overlay_rows_before = _safe_read_rows(OVERLAY_MANIFEST_PATH)
sequence_has_pose_before = any(_frame_pose_coverage(row) > 0.0 for row in structured_frame_rows_before)
overlay_has_pose_before = any(bool(row.get('includes_pose')) for row in overlay_rows_before)
DEEP_CV_POSE_INCOMPLETE = ENABLE_RTMPOSE and (
    deep_cv_status_before != 'complete'
    or pose_rows_before == 0
    or (deep_cv_resolved_clips_before > 0 and deep_cv_completed_pose_before < deep_cv_resolved_clips_before)
)
FORCE_DEEP_CV_FOR_POSE = DEEP_CV_POSE_INCOMPLETE
FORCE_DEEP_SEQUENCE_FOR_POSE = ENABLE_RTMPOSE and (FORCE_DEEP_CV_FOR_POSE or (pose_rows_before > 0 and not sequence_has_pose_before))
FORCE_OVERLAY_FOR_POSE = ENABLE_RTMPOSE and (FORCE_DEEP_CV_FOR_POSE or not overlay_has_pose_before)
if FORCE_DEEP_SEQUENCE_FOR_POSE:
    os.environ['FORCE_REBUILD_DEEP_SEQUENCE'] = '1'
else:
    os.environ.pop('FORCE_REBUILD_DEEP_SEQUENCE', None)

print('ENABLE_RTMPOSE =', ENABLE_RTMPOSE)
print('POSE_BACKEND =', POSE_BACKEND)
print('deep_cv_status_before =', deep_cv_status_before)
print('deep_cv_rows_before =', deep_cv_rows_before)
print('deep_cv_completed_pose_clips =', deep_cv_completed_pose_before, '/', deep_cv_resolved_clips_before)
print('pose_rows_before =', pose_rows_before)
print('sequence_has_pose_before =', sequence_has_pose_before)
print('overlay_has_pose_before =', overlay_has_pose_before)
print('force_deep_cv_for_pose =', FORCE_DEEP_CV_FOR_POSE)
print('force_deep_sequence_for_pose =', FORCE_DEEP_SEQUENCE_FOR_POSE)
print('force_overlay_for_pose =', FORCE_OVERLAY_FOR_POSE)

RUN_DEEP_CV = True
RUN_DEEP_SEQUENCE_FEATURES = True
RUN_CV_OVERLAY_VIDEOS = True

stages = [
    {
        'notebook': '16_deep_cv_yolo_pose_homography.ipynb',
        'label': 'YOLO/tracking/pose/bat/plate/homography artifacts',
        'enabled': RUN_DEEP_CV,
        'force': FORCE_RERUN_ALL or FORCE_DEEP_CV_FOR_POSE,
        'expected_outputs': [f'detections/{FULL_RUN_ID}/detections_v1.parquet', f'tracks/{FULL_RUN_ID}/tracks_v1.parquet', f'pose2d/{FULL_RUN_ID}/pose2d_v1.parquet', f'objects/{FULL_RUN_ID}/bat_detection_v1.parquet', f'objects/{FULL_RUN_ID}/bat_line_v1.parquet', f'homography/{FULL_RUN_ID}/homography_v1.parquet', f'reports/preflight/deep_cv_{FULL_RUN_ID}.json', f'reports/preflight/deep_cv_{FULL_RUN_ID}_progress.json'],
        'progress_files': [f'reports/preflight/deep_cv_{FULL_RUN_ID}_progress.json'],
    },
    {
        'notebook': '17_deep_sequence_features.ipynb',
        'label': 'Deep CV sequence features',
        'enabled': RUN_DEEP_SEQUENCE_FEATURES,
        'force': FORCE_RERUN_ALL or FORCE_DEEP_SEQUENCE_FOR_POSE,
        'expected_outputs': [f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/manifest.parquet', f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/frames.parquet', f'reports/preflight/deep_sequence_features_{FULL_RUN_ID}.json', f'reports/preflight/deep_sequence_features_{FULL_RUN_ID}_progress.json'],
        'progress_files': [f'reports/preflight/deep_sequence_features_{FULL_RUN_ID}_progress.json'],
    },
    {
        'notebook': '21_cv_overlay_videos.ipynb',
        'label': 'Visual overlay videos',
        'enabled': RUN_CV_OVERLAY_VIDEOS,
        'force': FORCE_RERUN_ALL or FORCE_OVERLAY_FOR_POSE,
        'expected_outputs': [f'debug/{FULL_RUN_ID}/cv_overlay_videos_v1.parquet', f'debug/{FULL_RUN_ID}/cv_overlays', f'debug/{FULL_RUN_ID}/debug_overlays_v1.parquet', f'reports/preflight/cv_overlay_videos_{FULL_RUN_ID}.json', f'reports/preflight/cv_overlay_videos_{FULL_RUN_ID}_progress.json'],
        'progress_files': [f'reports/preflight/cv_overlay_videos_{FULL_RUN_ID}_progress.json'],
    },
]
LOGGER.run_stages(ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, stages=stages)
